# Data Coverage Audit

Goal: Improve the quality and coverage of the AQI-weather dataset before feature engineering.

Current issue:
- Merged AQI-weather dataset has only 252 rows across 3 Delhi stations.
- Punjabi Bagh: 89 rows
- R K Puram: 88 rows
- Anand Vihar: 75 rows

We will check:
1. OpenAQ pagination
2. Raw vs hourly OpenAQ endpoint
3. Best Delhi PM2.5 stations by coverage
4. Whether 90 days improves row counts
5. Whether station-wise models are still realistic

In [2]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
import time

In [24]:
import os
from dotenv import load_dotenv

In [ ]:
OPENAQ_API_KEY = os.getenv("OPENAQ_API_KEY")

headers = {
    "X-API-Key": OPENAQ_API_KEY
}

In [25]:
current_aqi = pd.read_csv("../data/processed/clean_aqi_pm25.csv")

current_aqi.head()

,location_id,location_name,sensor_id,parameter,value,unit,datetime_utc,datetime_local,latitude,longitude,hour
0,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,105.0,µg/m³,2025-02-18 22:00:00+00:00,2025-02-19 03:30:00+05:30,28.646835,77.316032,2025-02-19 03:00:00+05:30
1,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 22:15:00+00:00,2025-02-19 03:45:00+05:30,28.646835,77.316032,2025-02-19 03:00:00+05:30
2,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 22:30:00+00:00,2025-02-19 04:00:00+05:30,28.646835,77.316032,2025-02-19 04:00:00+05:30
3,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 22:45:00+00:00,2025-02-19 04:15:00+05:30,28.646835,77.316032,2025-02-19 04:00:00+05:30
4,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 23:00:00+00:00,2025-02-19 04:30:00+05:30,28.646835,77.316032,2025-02-19 04:00:00+05:30


In [26]:
current_aqi.info()

<class 'pandas.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   location_id     252 non-null    int64  
 1   location_name   252 non-null    str    
 2   sensor_id       252 non-null    int64  
 3   parameter       252 non-null    str    
 4   value           252 non-null    float64
 5   unit            252 non-null    str    
 6   datetime_utc    252 non-null    str    
 7   datetime_local  252 non-null    str    
 8   latitude        252 non-null    float64
 9   longitude       252 non-null    float64
 10  hour            252 non-null    str    
dtypes: float64(3), int64(2), str(6)
memory usage: 21.8 KB


In [27]:
current_aqi["location_name"].value_counts()

location_name
Punjabi Bagh, Delhi - DPCC       89
R K Puram, Delhi - DPCC          88
Anand Vihar, New Delhi - DPCC    75
Name: count, dtype: int64

In [28]:
current_aqi[["location_name", "sensor_id"]].drop_duplicates()

,location_name,sensor_id
0,"Anand Vihar, New Delhi - DPCC",12235610
75,"Punjabi Bagh, Delhi - DPCC",12234796
164,"R K Puram, Delhi - DPCC",12234787


In [29]:
def fetch_openaq_page_1(sensor_id, date_from, date_to, limit=1000):
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements"
    
    params = {
        "datetime_from": date_from,
        "datetime_to": date_to,
        "limit": limit,
        "page": 1
    }
    
    response = requests.get(url, headers=headers, params=params)
    
    print("URL:", response.url)
    print("Status code:", response.status_code)
    
    response.raise_for_status()
    
    data = response.json()
    return data.get("results", [])

In [30]:
def fetch_openaq_all_pages(sensor_id, date_from, date_to, limit=1000, max_pages=20):
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements"
    
    all_results = []
    
    for page in range(1, max_pages + 1):
        params = {
            "datetime_from": date_from,
            "datetime_to": date_to,
            "limit": limit,
            "page": page
        }
        
        response = requests.get(url, headers=headers, params=params)
        
        print(f"Sensor {sensor_id} | Page {page} | Status {response.status_code}")
        
        response.raise_for_status()
        
        data = response.json()
        results = data.get("results", [])
        
        print(f"Rows on page {page}: {len(results)}")
        
        all_results.extend(results)
        
        # If we got less than the limit, this is probably the last page
        if len(results) < limit:
            break
        
        time.sleep(0.2)
    
    return all_results

In [31]:
date_to = datetime.now(timezone.utc)
date_from = date_to - timedelta(days=60)

date_from_str = date_from.strftime("%Y-%m-%dT%H:%M:%SZ")
date_to_str = date_to.strftime("%Y-%m-%dT%H:%M:%SZ")

date_from_str, date_to_str

('2026-04-04T19:45:50Z', '2026-06-03T19:45:50Z')

In [32]:
station_sensors = (
    current_aqi[["location_name", "sensor_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

station_sensors

,location_name,sensor_id
0,"Anand Vihar, New Delhi - DPCC",12235610
1,"Punjabi Bagh, Delhi - DPCC",12234796
2,"R K Puram, Delhi - DPCC",12234787


In [33]:
audit_rows = []

for _, row in station_sensors.iterrows():
    location_name = row["location_name"]
    sensor_id = row["sensor_id"]
    
    print("=" * 60)
    print(location_name, sensor_id)
    
    page_1_results = fetch_openaq_page_1(
        sensor_id=sensor_id,
        date_from=date_from_str,
        date_to=date_to_str,
        limit=1000
    )
    
    all_page_results = fetch_openaq_all_pages(
        sensor_id=sensor_id,
        date_from=date_from_str,
        date_to=date_to_str,
        limit=1000,
        max_pages=20
    )
    
    audit_rows.append({
        "location_name": location_name,
        "sensor_id": sensor_id,
        "page_1_rows": len(page_1_results),
        "all_pages_rows": len(all_page_results),
        "pagination_added_rows": len(all_page_results) - len(page_1_results),
        "page_1_hit_limit": len(page_1_results) == 1000
    })

pagination_audit = pd.DataFrame(audit_rows)
pagination_audit


Anand Vihar, New Delhi - DPCC 12235610
URL: https://api.openaq.org/v3/sensors/12235610/measurements?datetime_from=2026-04-04T19%3A45%3A50Z&datetime_to=2026-06-03T19%3A45%3A50Z&limit=1000&page=1
Status code: 200
Sensor 12235610 | Page 1 | Status 200
Rows on page 1: 1000
Sensor 12235610 | Page 2 | Status 200
Rows on page 2: 1000
Sensor 12235610 | Page 3 | Status 200
Rows on page 3: 1000
Sensor 12235610 | Page 4 | Status 200
Rows on page 4: 1000
Sensor 12235610 | Page 5 | Status 200
Rows on page 5: 445
Punjabi Bagh, Delhi - DPCC 12234796
URL: https://api.openaq.org/v3/sensors/12234796/measurements?datetime_from=2026-04-04T19%3A45%3A50Z&datetime_to=2026-06-03T19%3A45%3A50Z&limit=1000&page=1
Status code: 200
Sensor 12234796 | Page 1 | Status 200
Rows on page 1: 1000
Sensor 12234796 | Page 2 | Status 200
Rows on page 2: 1000
Sensor 12234796 | Page 3 | Status 200
Rows on page 3: 1000
Sensor 12234796 | Page 4 | Status 200
Rows on page 4: 1000
Sensor 12234796 | Page 5 | Status 200
Rows on page 

,location_name,sensor_id,page_1_rows,all_pages_rows,pagination_added_rows,page_1_hit_limit
0,"Anand Vihar, New Delhi - DPCC",12235610,1000,4445,3445,True
1,"Punjabi Bagh, Delhi - DPCC",12234796,1000,4674,3674,True
2,"R K Puram, Delhi - DPCC",12234787,1000,4769,3769,True


In [34]:
pagination_audit.to_csv("../data/processed/openaq_pagination_audit.csv", index=False)

---

In [35]:
def fetch_openaq_hourly(sensor_id, date_from, date_to, limit=1000, max_pages=10):
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/hours"
    
    all_results = []
    
    for page in range(1, max_pages + 1):
        params = {
            "datetime_from": date_from,
            "datetime_to": date_to,
            "limit": limit,
            "page": page
        }
        
        response = requests.get(url, headers=headers, params=params)
        print(f"Sensor {sensor_id} | Hourly page {page} | Status {response.status_code}")
        
        response.raise_for_status()
        
        data = response.json()
        results = data.get("results", [])
        
        print(f"Hourly rows on page {page}: {len(results)}")
        
        all_results.extend(results)
        
        if len(results) < limit:
            break
        
        time.sleep(0.2)
    
    return all_results

In [36]:
hourly_audit_rows = []
all_hourly_results = []

for _, row in station_sensors.iterrows():
    location_name = row["location_name"]
    sensor_id = row["sensor_id"]
    
    print("=" * 60)
    print(location_name, sensor_id)
    
    hourly_results = fetch_openaq_hourly(
        sensor_id=sensor_id,
        date_from=date_from_str,
        date_to=date_to_str,
        limit=1000,
        max_pages=10
    )
    
    hourly_audit_rows.append({
        "location_name": location_name,
        "sensor_id": sensor_id,
        "hourly_rows": len(hourly_results),
        "expected_60_day_hourly_rows": 60 * 24,
        "coverage_percent": round((len(hourly_results) / (60 * 24)) * 100, 2)
    })
    
    for item in hourly_results:
        item["location_name"] = location_name
        item["sensor_id"] = sensor_id
        all_hourly_results.append(item)

hourly_audit = pd.DataFrame(hourly_audit_rows)
hourly_audit

Anand Vihar, New Delhi - DPCC 12235610
Sensor 12235610 | Hourly page 1 | Status 200
Hourly rows on page 1: 1000
Sensor 12235610 | Hourly page 2 | Status 200
Hourly rows on page 2: 263
Punjabi Bagh, Delhi - DPCC 12234796
Sensor 12234796 | Hourly page 1 | Status 200
Hourly rows on page 1: 1000
Sensor 12234796 | Hourly page 2 | Status 200
Hourly rows on page 2: 284
R K Puram, Delhi - DPCC 12234787
Sensor 12234787 | Hourly page 1 | Status 200
Hourly rows on page 1: 1000
Sensor 12234787 | Hourly page 2 | Status 200
Hourly rows on page 2: 317


,location_name,sensor_id,hourly_rows,expected_60_day_hourly_rows,coverage_percent
0,"Anand Vihar, New Delhi - DPCC",12235610,1263,1440,87.71
1,"Punjabi Bagh, Delhi - DPCC",12234796,1284,1440,89.17
2,"R K Puram, Delhi - DPCC",12234787,1317,1440,91.46


In [37]:
hourly_df = pd.json_normalize(all_hourly_results)

hourly_df.head()

,value,coordinates,location_name,sensor_id,flagInfo.hasFlags,parameter.id,parameter.name,parameter.units,parameter.displayName,period.label,...,coverage.expectedCount,coverage.expectedInterval,coverage.observedCount,coverage.observedInterval,coverage.percentComplete,coverage.percentCoverage,coverage.datetimeFrom.utc,coverage.datetimeFrom.local,coverage.datetimeTo.utc,coverage.datetimeTo.local
0,50.0,None,"Anand Vihar, New Delhi - DPCC",12235610,False,2,pm25,µg/m³,None,1hour,...,4,01:00:00,3,00:45:00,75.0,75.0,2026-04-04T18:45:00Z,2026-04-05T00:15:00+05:30,2026-04-04T20:15:00Z,2026-04-05T01:45:00+05:30
1,75.5,None,"Anand Vihar, New Delhi - DPCC",12235610,False,2,pm25,µg/m³,None,1hour,...,4,01:00:00,4,01:00:00,100.0,100.0,2026-04-04T19:45:00Z,2026-04-05T01:15:00+05:30,2026-04-04T21:30:00Z,2026-04-05T03:00:00+05:30
2,44.0,None,"Anand Vihar, New Delhi - DPCC",12235610,False,2,pm25,µg/m³,None,1hour,...,4,01:00:00,2,00:30:00,50.0,50.0,2026-04-04T21:00:00Z,2026-04-05T02:30:00+05:30,2026-04-04T22:15:00Z,2026-04-05T03:45:00+05:30
3,39.8,None,"Anand Vihar, New Delhi - DPCC",12235610,False,2,pm25,µg/m³,None,1hour,...,4,01:00:00,4,01:00:00,100.0,100.0,2026-04-04T21:45:00Z,2026-04-05T03:15:00+05:30,2026-04-04T23:30:00Z,2026-04-05T05:00:00+05:30
4,57.3,None,"Anand Vihar, New Delhi - DPCC",12235610,False,2,pm25,µg/m³,None,1hour,...,4,01:00:00,4,01:00:00,100.0,100.0,2026-04-04T22:45:00Z,2026-04-05T04:15:00+05:30,2026-04-05T00:30:00Z,2026-04-05T06:00:00+05:30


In [38]:
hourly_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3864 entries, 0 to 3863
Data columns (total 34 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   value                        3864 non-null   float64
 1   coordinates                  0 non-null      object 
 2   location_name                3864 non-null   str    
 3   sensor_id                    3864 non-null   int64  
 4   flagInfo.hasFlags            3864 non-null   bool   
 5   parameter.id                 3864 non-null   int64  
 6   parameter.name               3864 non-null   str    
 7   parameter.units              3864 non-null   str    
 8   parameter.displayName        0 non-null      object 
 9   period.label                 3864 non-null   str    
 10  period.interval              3864 non-null   str    
 11  period.datetimeFrom.utc      3864 non-null   str    
 12  period.datetimeFrom.local    3864 non-null   str    
 13  period.datetimeTo.utc        

In [39]:
hourly_df.columns

Index(['value', 'coordinates', 'location_name', 'sensor_id',
       'flagInfo.hasFlags', 'parameter.id', 'parameter.name',
       'parameter.units', 'parameter.displayName', 'period.label',
       'period.interval', 'period.datetimeFrom.utc',
       'period.datetimeFrom.local', 'period.datetimeTo.utc',
       'period.datetimeTo.local', 'summary.min', 'summary.q02', 'summary.q25',
       'summary.median', 'summary.q75', 'summary.q98', 'summary.max',
       'summary.avg', 'summary.sd', 'coverage.expectedCount',
       'coverage.expectedInterval', 'coverage.observedCount',
       'coverage.observedInterval', 'coverage.percentComplete',
       'coverage.percentCoverage', 'coverage.datetimeFrom.utc',
       'coverage.datetimeFrom.local', 'coverage.datetimeTo.utc',
       'coverage.datetimeTo.local'],
      dtype='str')

In [40]:
hourly_audit.to_csv("../data/processed/openaq_hourly_coverage_audit.csv", index=False)

In [ ]:
hourly_audit


,location_name,sensor_id,hourly_rows,expected_60_day_hourly_rows,coverage_percent
0,"Anand Vihar, New Delhi - DPCC",12235610,1263,1440,87.71
1,"Punjabi Bagh, Delhi - DPCC",12234796,1284,1440,89.17
2,"R K Puram, Delhi - DPCC",12234787,1317,1440,91.46


---
### Cleaning the AQI file

In [42]:
hourly_df.columns

Index(['value', 'coordinates', 'location_name', 'sensor_id',
       'flagInfo.hasFlags', 'parameter.id', 'parameter.name',
       'parameter.units', 'parameter.displayName', 'period.label',
       'period.interval', 'period.datetimeFrom.utc',
       'period.datetimeFrom.local', 'period.datetimeTo.utc',
       'period.datetimeTo.local', 'summary.min', 'summary.q02', 'summary.q25',
       'summary.median', 'summary.q75', 'summary.q98', 'summary.max',
       'summary.avg', 'summary.sd', 'coverage.expectedCount',
       'coverage.expectedInterval', 'coverage.observedCount',
       'coverage.observedInterval', 'coverage.percentComplete',
       'coverage.percentCoverage', 'coverage.datetimeFrom.utc',
       'coverage.datetimeFrom.local', 'coverage.datetimeTo.utc',
       'coverage.datetimeTo.local'],
      dtype='str')

In [43]:
hourly_df.head()

,value,coordinates,location_name,sensor_id,flagInfo.hasFlags,parameter.id,parameter.name,parameter.units,parameter.displayName,period.label,...,coverage.expectedCount,coverage.expectedInterval,coverage.observedCount,coverage.observedInterval,coverage.percentComplete,coverage.percentCoverage,coverage.datetimeFrom.utc,coverage.datetimeFrom.local,coverage.datetimeTo.utc,coverage.datetimeTo.local
0,50.0,None,"Anand Vihar, New Delhi - DPCC",12235610,False,2,pm25,µg/m³,None,1hour,...,4,01:00:00,3,00:45:00,75.0,75.0,2026-04-04T18:45:00Z,2026-04-05T00:15:00+05:30,2026-04-04T20:15:00Z,2026-04-05T01:45:00+05:30
1,75.5,None,"Anand Vihar, New Delhi - DPCC",12235610,False,2,pm25,µg/m³,None,1hour,...,4,01:00:00,4,01:00:00,100.0,100.0,2026-04-04T19:45:00Z,2026-04-05T01:15:00+05:30,2026-04-04T21:30:00Z,2026-04-05T03:00:00+05:30
2,44.0,None,"Anand Vihar, New Delhi - DPCC",12235610,False,2,pm25,µg/m³,None,1hour,...,4,01:00:00,2,00:30:00,50.0,50.0,2026-04-04T21:00:00Z,2026-04-05T02:30:00+05:30,2026-04-04T22:15:00Z,2026-04-05T03:45:00+05:30
3,39.8,None,"Anand Vihar, New Delhi - DPCC",12235610,False,2,pm25,µg/m³,None,1hour,...,4,01:00:00,4,01:00:00,100.0,100.0,2026-04-04T21:45:00Z,2026-04-05T03:15:00+05:30,2026-04-04T23:30:00Z,2026-04-05T05:00:00+05:30
4,57.3,None,"Anand Vihar, New Delhi - DPCC",12235610,False,2,pm25,µg/m³,None,1hour,...,4,01:00:00,4,01:00:00,100.0,100.0,2026-04-04T22:45:00Z,2026-04-05T04:15:00+05:30,2026-04-05T00:30:00Z,2026-04-05T06:00:00+05:30


In [45]:
clean_hourly_aqi = hourly_df.copy()

clean_hourly_aqi = clean_hourly_aqi.rename(columns={
    "period.datetimeFrom.utc": "datetime",
    "value": "pm25",
    "parameter.name": "parameter"
})

clean_hourly_aqi = clean_hourly_aqi[
    [
        "location_name",
        "sensor_id",
        "datetime",
        "parameter",
        "pm25",
        "coordinates",
        "coverage.percentComplete",
        "coverage.percentCoverage"
    ]
]

clean_hourly_aqi.head()

,location_name,sensor_id,datetime,parameter,pm25,coordinates,coverage.percentComplete,coverage.percentCoverage
0,"Anand Vihar, New Delhi - DPCC",12235610,2026-04-04T19:30:00Z,pm25,50.0,None,75.0,75.0
1,"Anand Vihar, New Delhi - DPCC",12235610,2026-04-04T20:30:00Z,pm25,75.5,None,100.0,100.0
2,"Anand Vihar, New Delhi - DPCC",12235610,2026-04-04T21:30:00Z,pm25,44.0,None,50.0,50.0
3,"Anand Vihar, New Delhi - DPCC",12235610,2026-04-04T22:30:00Z,pm25,39.8,None,100.0,100.0
4,"Anand Vihar, New Delhi - DPCC",12235610,2026-04-04T23:30:00Z,pm25,57.3,None,100.0,100.0


In [46]:
clean_hourly_aqi["datetime"] = pd.to_datetime(clean_hourly_aqi["datetime"], utc=True)
clean_hourly_aqi["pm25"] = pd.to_numeric(clean_hourly_aqi["pm25"], errors="coerce")

clean_hourly_aqi.info()

<class 'pandas.DataFrame'>
RangeIndex: 3864 entries, 0 to 3863
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   location_name             3864 non-null   str                
 1   sensor_id                 3864 non-null   int64              
 2   datetime                  3864 non-null   datetime64[us, UTC]
 3   parameter                 3864 non-null   str                
 4   pm25                      3864 non-null   float64            
 5   coordinates               0 non-null      object             
 6   coverage.percentComplete  3864 non-null   float64            
 7   coverage.percentCoverage  3864 non-null   float64            
dtypes: datetime64[us, UTC](1), float64(3), int64(1), object(1), str(2)
memory usage: 241.6+ KB


In [47]:
clean_hourly_aqi = clean_hourly_aqi.sort_values(
    ["location_name", "datetime"]
)

clean_hourly_aqi = clean_hourly_aqi.drop_duplicates(
    subset=["location_name", "sensor_id", "datetime"]
)

clean_hourly_aqi.shape

(3864, 8)

In [48]:
clean_hourly_aqi["location_name"].value_counts()

location_name
R K Puram, Delhi - DPCC          1317
Punjabi Bagh, Delhi - DPCC       1284
Anand Vihar, New Delhi - DPCC    1263
Name: count, dtype: int64

In [49]:
clean_hourly_aqi.isna().sum()

location_name                  0
sensor_id                      0
datetime                       0
parameter                      0
pm25                           0
coordinates                 3864
coverage.percentComplete       0
coverage.percentCoverage       0
dtype: int64

In [50]:
clean_hourly_aqi[["pm25", "coordinates"]].isna().sum()

pm25              0
coordinates    3864
dtype: int64

In [56]:
clean_hourly_aqi.isna().sum()

location_name               0
sensor_id                   0
datetime                    0
parameter                   0
pm25                        0
coverage.percentComplete    0
coverage.percentCoverage    0
dtype: int64

In [57]:
clean_hourly_aqi["pm25"].isna().sum()

np.int64(0)

In [58]:
clean_hourly_aqi.to_csv(
    "../data/processed/clean_aqi_pm25_hourly.csv",
    index=False
)

In [59]:
pd.read_csv("../data/processed/clean_aqi_pm25_hourly.csv").shape

(3864, 7)

In [60]:
pd.read_csv("../data/processed/clean_aqi_pm25_hourly.csv")["location_name"].value_counts()

location_name
R K Puram, Delhi - DPCC          1317
Punjabi Bagh, Delhi - DPCC       1284
Anand Vihar, New Delhi - DPCC    1263
Name: count, dtype: int64

In [61]:
print("pm25 missing:", clean_hourly_aqi["pm25"].isna().sum())
print("shape:", clean_hourly_aqi.shape)
print("\nstation counts:")
print(clean_hourly_aqi["location_name"].value_counts())

pm25 missing: 0
shape: (3864, 7)

station counts:
location_name
R K Puram, Delhi - DPCC          1317
Punjabi Bagh, Delhi - DPCC       1284
Anand Vihar, New Delhi - DPCC    1263
Name: count, dtype: int64
